## [G] Business Cycle Analysis

In [ ]:
import pandas as pd
import numpy as np
import os
import statsmodels.api as sm
from scipy.stats import binomtest

In [2]:
path = "data/processed_data/"
file_name = "03_bc_data.csv"

date_column = "date" 
stock_column = "stock"

number_of_portfolios = 5

# Set path
os.chdir("..")
print(os.getcwd())

/workspaces/master-thesis-submission/src


In [3]:
bc_data = pd.read_csv(path + file_name)

In [4]:
bc_data.head()

,date,stock,stock_return,esg_momentum_score,mcap,Mkt-RF,SMB,HML,RMW,CMA,RF,WML,dividend_yield_ttm,term_spread,default_spread,spot_rate_3m_government_bond,recession_indicator
0,2016-06-30,A2.MI,-0.080406,-1.622974,3.617623e+09,-0.0501,-0.0224,-0.0164,0.0325,0.0282,0.0002,0.071,0.045862,0.005456,0.0103,-0.0065,0
1,2016-06-30,AAAA.L^C21,-0.232982,2.922803,1.749150e+09,-0.0501,-0.0224,-0.0164,0.0325,0.0282,0.0002,0.071,0.045862,0.005456,0.0103,-0.0065,0
2,2016-06-30,AAL.L,0.115835,-7.452243,1.221423e+10,-0.0501,-0.0224,-0.0164,0.0325,0.0282,0.0002,0.071,0.045862,0.005456,0.0103,-0.0065,0
3,2016-06-30,AALB.AS,-0.146957,5.607441,2.991192e+09,-0.0501,-0.0224,-0.0164,0.0325,0.0282,0.0002,0.071,0.045862,0.005456,0.0103,-0.0065,0
4,2016-06-30,ABBN.S,-0.056730,-3.994408,3.818451e+10,-0.0501,-0.0224,-0.0164,0.0325,0.0282,0.0002,0.071,0.045862,0.005456,0.0103,-0.0065,0


## 1. Calculate ESG Momentum Portfolios

### 1.1 General Helper Functions

In [5]:
def assign_monthly_quantile_portfolios(
    df_raw,
    date_col= date_column,
    score_col= "esg_momentum_score",
    n_portfolios = number_of_portfolios,
    portfolio_col = "portfolio",
    dropna_cols = (date_column, stock_column, "esg_momentum_score"),
    ):

    # Make a copy of the original DataFrame to avoid modifying it directly
    df = df_raw.copy()

    # Convert the date column to datetime format
    df[date_col] = pd.to_datetime(df[date_col])

    # Drop rows with missing values in the specified columns
    df = df.dropna(subset=list(dropna_cols))

    def _assign(g):

        # Make a copy of the group to avoid modifying it directly
        gg = g.copy()

        # Rank the stocks based on the score column
        gg["_rank"] = gg[score_col].rank(method="first", ascending=True)

        # Assign quantile portfolios based on the rank
        gg[portfolio_col] = pd.qcut(gg["_rank"], q=n_portfolios, labels=list(range(1, n_portfolios + 1))).astype(int)

        # Drop the temporary rank column before returning the group
        return gg.drop(columns=["_rank"])

    # Group the DataFrame by the date column and apply the _assign function to each group
    # group_keys = False is used to prevent the date column from being added back as an index level after the groupby operation
    df = df.groupby(date_col, group_keys=False).apply(_assign)

    return df

In [6]:
def build_portfolio_constituents_table(
    df_with_portfolios,
    date_col= "date",
    stock_col = "stock",
    portfolio_col = "portfolio",
    n_portfolios = number_of_portfolios,
    prefix = "portfolio_",
):
    
    # Make a copy of the DataFrame to avoid modifying the original one
    df = df_with_portfolios.copy()

    # Convert the date column to datetime format to ensure proper sorting and grouping
    df[date_col] = pd.to_datetime(df[date_col])

    # Sort the DataFrame by date, portfolio, and stock, then group by date and portfolio to create lists of stocks for each portfolio on each date
    out = (
        df.sort_values([date_col, portfolio_col, stock_col]) # Sort the DataFrame by date, portfolio, and stock to ensure consistent ordering
         .groupby([date_col, portfolio_col])[stock_col] # Group by date and portfolio, then select the stock column
         .apply(list) # Convert the grouped stocks into lists
         .unstack(portfolio_col) # Unstack the portfolio column to create separate columns for each portfolio
         .rename(columns={i: f"{prefix}{i}_constituents" for i in range(1, n_portfolios + 1)}) # Rename the cols
         .reset_index() # Reset the index to turn the date back into a column
         .sort_values(date_col) # Sort the DataFrame by date to ensure chronological order
         .reset_index(drop=True) # Reset the index again to have a clean integer index after sorting
    )

    # Create a count DataFrame to count the number of stocks in each portfolio for each date
    count_df = df_with_portfolios.groupby([date_col, portfolio_col]).size().unstack(portfolio_col).fillna(0).astype(int)

    # Name columns portfolio_x_count to indicate they represent counts of stocks in each portfolio
    count_df.columns = [f"{prefix}{col}_count" for col in count_df.columns]

    # Merge the count DataFrame with the original output to include the counts of stocks in each portfolio
    out = out.merge(count_df, on=date_col, suffixes=("", "_count"))

    return out

In [7]:
def calculate_newey_west_stats(values, maxlags=None):
    x = pd.Series(values).dropna()
    n = len(x)

    if maxlags is None:
        maxlags = int(np.floor(4 * (n / 100) ** (2 / 9)))

    # ensure maxlags is valid
    maxlags = min(maxlags, n - 1)

    # constant-only regression: mean payoff = intercept
    y = x.astype(float)
    X = np.ones(n)

    model = sm.OLS(y, X).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": maxlags}
    )

    return pd.DataFrame([{
        "mean": model.params[0],
        "std_dev": x.std(ddof=1),
        "t_statistic": model.tvalues[0],
        "p_value": round(model.pvalues[0], 4),
        "degrees_of_freedom": n - 1,
        "n": n,
        "max_lags": maxlags
    }])

### 1.2 Base Calculations & Analysis

In [8]:
# Assign each stock to a portfolio based on esg momentum.
portfolios = assign_monthly_quantile_portfolios(bc_data)
portfolios.to_csv("data/processed_data/07_portfolios.csv", index=False)

# Create table with lists of constituents per portfolio per month & export as csv
portfolio_groups = build_portfolio_constituents_table(portfolios)
portfolio_groups.to_csv("data/processed_data/07_portfolio_constituents.csv", index=False)

portfolios.head()

/tmp/ipykernel_697/1779438100.py:35: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(date_col, group_keys=False).apply(_assign)


,date,stock,stock_return,esg_momentum_score,mcap,Mkt-RF,SMB,HML,RMW,CMA,RF,WML,dividend_yield_ttm,term_spread,default_spread,spot_rate_3m_government_bond,recession_indicator,portfolio
0,2016-06-30,A2.MI,-0.080406,-1.622974,3.617623e+09,-0.0501,-0.0224,-0.0164,0.0325,0.0282,0.0002,0.071,0.045862,0.005456,0.0103,-0.0065,0,1
1,2016-06-30,AAAA.L^C21,-0.232982,2.922803,1.749150e+09,-0.0501,-0.0224,-0.0164,0.0325,0.0282,0.0002,0.071,0.045862,0.005456,0.0103,-0.0065,0,3
2,2016-06-30,AAL.L,0.115835,-7.452243,1.221423e+10,-0.0501,-0.0224,-0.0164,0.0325,0.0282,0.0002,0.071,0.045862,0.005456,0.0103,-0.0065,0,1
3,2016-06-30,AALB.AS,-0.146957,5.607441,2.991192e+09,-0.0501,-0.0224,-0.0164,0.0325,0.0282,0.0002,0.071,0.045862,0.005456,0.0103,-0.0065,0,4
4,2016-06-30,ABBN.S,-0.056730,-3.994408,3.818451e+10,-0.0501,-0.0224,-0.0164,0.0325,0.0282,0.0002,0.071,0.045862,0.005456,0.0103,-0.0065,0,1


## 2. Analysis

### 2.1 ESG Momentum Payoffs Classified by Business Cycles

In [ ]:
def compute_next_month_portfolio_returns(
    df_with_portfolios,
    date_col = "date",
    stock_col = "stock",
    ret_col = "stock_return",
    mcap_col = "mcap",
    portfolio_col = "portfolio",
    n_portfolios = number_of_portfolios,
    month_end_shift = 1
):

    df = df_with_portfolios.copy()
    df[date_col] = pd.to_datetime(df[date_col])

    # Portfolio membership at formation month t
    membership = df[[date_col, stock_col, portfolio_col, mcap_col]].copy()
    membership = membership.rename(columns={date_col: "formation_date", mcap_col: "mcap_formation"})
    membership["holding_date"] = membership["formation_date"] + pd.offsets.MonthEnd(month_end_shift)

    # Holding-month returns at t+1
    next_returns = df[[date_col, stock_col, ret_col]].copy()
    next_returns = next_returns.rename(columns={date_col: "holding_date", ret_col: "holding_return"})
    next_returns = next_returns.dropna(subset=["holding_return"])

    # Merge membership with holding-month returns
    merged = membership.merge(next_returns, on=["holding_date", stock_col], how="inner")

    # Helper function to compute equal-weighted and value-weighted returns for each portfolio, handling cases with no valid observations for value-weighting
    def _agg(g: pd.DataFrame) -> pd.Series:

        # Drop rows with missing values in the formation MCAP column to ensure I only compute returns for stocks with valid MCAP at formation
        g_subset = g.dropna(subset=["mcap_formation"]).copy()
        g_subset = g_subset[g_subset["mcap_formation"] > 0] # Ensure I only consider stocks with positive MCAP for return calculations

        if g_subset.empty:
            ew = np.nan
            vw = np.nan
        else:
            ew = g_subset["holding_return"].mean()
            vw = np.average(g_subset["holding_return"], weights=g_subset["mcap_formation"])

        return pd.Series({"ew_return": ew, "vw_return": vw})

    # Group by holding date and portfolio, then apply the aggregation function to compute returns for each portfolio on each date
    long = merged.groupby(["holding_date", portfolio_col]).apply(_agg).reset_index()

    # Pivot the long format DataFrame to have one row per date and separate columns for each portfolio's equal-weighted and value-weighted returns
    ew_wide = long.pivot(index="holding_date", columns=portfolio_col, values="ew_return")
    vw_wide = long.pivot(index="holding_date", columns=portfolio_col, values="vw_return")

    # Create the final output DataFrame with the date column and the returns for each portfolio
    out = pd.DataFrame({"date": ew_wide.index}).reset_index(drop=True)

    # Add equal-weighted and value-weighted returns for each portfolio to the output DataFrame
    for i in range(1, n_portfolios + 1):
        out[f"ew_portfolio_{i}"] = ew_wide.get(i).values if i in ew_wide.columns else np.nan

    # Add value-weighted returns for each portfolio to the output DataFrame
    for i in range(1, n_portfolios + 1):
        out[f"vw_portfolio_{i}"] = vw_wide.get(i).values if i in vw_wide.columns else np.nan

    # Sort the output DataFrame by date and reset the index to ensure a clean integer index
    out = out.sort_values("date").reset_index(drop=True)

    return out

In [ ]:
# Compute next-month returns for each portfolio and get recession indicator
recession_indicator = bc_data[["date", "recession_indicator"]].drop_duplicates().reset_index(drop=True)
portfolio_returns = compute_next_month_portfolio_returns(portfolios)
portfolio_returns.to_csv("data/processed_data/07_portfolio_returns.csv", index=False)

# Convert date columns to datetime format
recession_indicator[date_column] = pd.to_datetime(recession_indicator[date_column]).dt.date
portfolio_returns[date_column] = pd.to_datetime(portfolio_returns[date_column]).dt.date

# Merge the portfolio returns with the recession indicator on the date column, using a left join to keep all rows from portfolio_returns and add the recession indicator where available
merged_returns = portfolio_returns.merge(recession_indicator, left_on="date", right_on="date", how="left")

# Drop rows with missing values in the recession_indicator column to ensure that I only have complete data for analysis
merged_returns.dropna(subset=["recession_indicator"], inplace=True)

/tmp/ipykernel_697/3793592047.py:45: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  long = merged.groupby(["holding_date", portfolio_col]).apply(_agg).reset_index()


In [11]:
merged_returns.head()

,date,ew_portfolio_1,ew_portfolio_2,ew_portfolio_3,ew_portfolio_4,ew_portfolio_5,vw_portfolio_1,vw_portfolio_2,vw_portfolio_3,vw_portfolio_4,vw_portfolio_5,recession_indicator
0,2016-07-31,0.058798,0.040478,0.040174,0.052686,0.050841,0.040771,0.022913,0.044135,0.041525,0.026098,0
1,2016-08-31,0.010269,0.022324,0.010761,0.009992,0.013836,-0.000638,0.009098,0.013071,0.003618,-0.017773,0
2,2016-09-30,0.002111,-0.001214,-0.004239,0.010101,-0.009488,0.006220,0.005783,-0.005043,0.007480,-0.020312,0
3,2016-10-31,-0.003670,-0.011410,-0.003480,-0.013959,-0.029918,-0.015699,-0.008637,0.005504,-0.017255,-0.012340,0
4,2016-11-30,0.007509,0.008852,0.006412,0.020809,0.028716,0.000529,0.012205,0.005121,0.013444,0.021145,0


In [18]:
# For each portfolio ("ew_portfolio_1", "ew_portfolio_2", ..., "vw_portfolio_5"), compute the average return during recession and non-recession periods and save in dataframe
avg_returns_by_recession = []

for col in merged_returns.columns:
    if col.startswith("ew_portfolio_") or col.startswith("vw_portfolio_"):

        # Calculate the average return for recession and non-recession periods for the current portfolio column
        recession_avg = merged_returns[merged_returns["recession_indicator"] == 1][col].mean()
        non_recession_avg = merged_returns[merged_returns["recession_indicator"] == 0][col].mean()

        # Append the results to the avg_returns_by_recession list as a dictionary containing the portfolio name and the average returns for recession and non-recession periods
        avg_returns_by_recession.append({
            "portfolio": col,
            "recession_avg": recession_avg,
            "non_recession_avg": non_recession_avg
        })

avg_returns_by_recession_df = pd.DataFrame(avg_returns_by_recession)

avg_returns_by_recession_df.head(10)

,portfolio,recession_avg,non_recession_avg
0,ew_portfolio_1,-0.026436,0.006595
1,ew_portfolio_2,-0.027540,0.007300
2,ew_portfolio_3,-0.025504,0.007055
3,ew_portfolio_4,-0.021125,0.007355
4,ew_portfolio_5,-0.017665,0.008082
5,vw_portfolio_1,-0.033086,0.006768
6,vw_portfolio_2,-0.021609,0.007787
7,vw_portfolio_3,-0.017620,0.005967
8,vw_portfolio_4,-0.016635,0.006087
9,vw_portfolio_5,-0.016685,0.008923


In [22]:
for weight_type in ["ew", "vw"]:
    top_portfolio_col = f"{weight_type}_portfolio_{number_of_portfolios}"
    portfolio_1_col = f"{weight_type}_portfolio_1"

    # Calculate the esg momentum payoff as the difference between the returns of portfolio 5 and portfolio 1 for the current weight type
    merged_returns[f"{weight_type}_esg_momentum_payoff"] = merged_returns[top_portfolio_col] - merged_returns[portfolio_1_col]

In [23]:
def get_nw_lag(x):
    # Newey-West lag selection based on the formula: lag = 4 * (n / 100)^(2/9), 
    # where n is the number of observations after dropping NA values.
    # The lag is capped at n - 1 to ensure it does not exceed the number of available observations.
    # See Wooldrige (2013) p. 432 for details.
    n = x.dropna().shape[0]

    if n <= 1:
        return 0
    
    # Calculate the Newey-West lag using the formula and cap it at n - 1
    return min(
        int(np.floor(4 * (n / 100) ** (2 / 9))),
        n - 1
    )

def calculate_stats_with_lag(x):
    # Drop NA values
    x_clean = x.dropna()

    # Determine the appropriate Newey-West lag based on the number of observations in x_clean
    max_lags = get_nw_lag(x_clean)

    # Calculate Newey-West adjusted statistics for the input series x_clean using the determined max_lags for autocorrelation adjustment
    stats = calculate_newey_west_stats(x_clean, maxlags=max_lags)

    stats = stats.copy()
    stats["max_lags"] = max_lags # Add the max_lags used in the Newey-West adjustment to the stats DataFrame for reference

    return stats

In [24]:
detailed_time_periods = merged_returns.sort_values("date").copy()
detailed_time_periods["date"] = pd.to_datetime(detailed_time_periods["date"])

# Creates a new period every time recession_indicator changes
detailed_time_periods["period_id"] = detailed_time_periods["recession_indicator"].ne(detailed_time_periods["recession_indicator"].shift()).cumsum()

# Calculate the start and end date for each period_id, along with the recession indicator for that period
period_info = (
    detailed_time_periods.groupby("period_id", as_index=False)
      .agg(
            recession_indicator=("recession_indicator", "first"),
            start_date=("date", "first"),
            end_date=("date", "last")
      )
)

# Merge the period information back to the detailed_time_periods DataFrame
detailed_time_periods = detailed_time_periods.merge(
    period_info[["period_id", "start_date", "end_date"]],
    on="period_id",
    how="left"
)

# Calculate ESG momentum payoff averages for each weight type and each contiguous period
results = []

# Loop through each weight type (equal-weighted and value-weighted) to calculate the average ESG momentum payoff 
# for each contiguous period defined by the recession indicator, and store the results in a list
for weight_type in ["ew", "vw"]:
    payoff_col = f"{weight_type}_esg_momentum_payoff"

    temp = (
        detailed_time_periods.groupby("period_id")[payoff_col]
        .apply(calculate_stats_with_lag)
        .reset_index()
    )

    temp = temp.merge(period_info, on="period_id", how="left")
    temp["weight_type"] = weight_type

    temp = temp.rename(columns={
        "mean": "avg_esg_momentum_payoff"
    })

    results.append(temp)
    
esg_momentum_payoff_df = pd.concat(results, ignore_index=True)

# Reorder columns
esg_momentum_payoff_df = esg_momentum_payoff_df[
    [
        "weight_type",
        "period_id",
        "recession_indicator",
        "start_date",
        "end_date",
        "n",
        "avg_esg_momentum_payoff",
        "std_dev",
        "t_statistic",
        "p_value",
        "degrees_of_freedom",
        "max_lags"
    ]
]

esg_momentum_payoff_df

/tmp/ipykernel_697/3769629738.py:21: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  "mean": model.params[0],
/tmp/ipykernel_697/3769629738.py:23: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  "t_statistic": model.tvalues[0],
/tmp/ipykernel_697/3769629738.py:24: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  "p_value": round(model.pvalues[0], 4),
/tmp/ipykernel_697/3769629738.py:21: FutureWarning: Series.__getitem__ treating keys as po

,weight_type,period_id,recession_indicator,start_date,end_date,n,avg_esg_momentum_payoff,std_dev,t_statistic,p_value,degrees_of_freedom,max_lags
0,ew,1,0,2016-07-31,2019-12-31,42,0.002366,0.011939,1.115568,0.2646,41,3
1,ew,2,1,2020-01-31,2020-06-30,6,0.008770,0.014592,2.180187,0.0292,5,2
2,ew,3,0,2020-07-31,2025-09-30,63,0.000900,0.012886,0.478243,0.6325,62,3
3,vw,1,0,2016-07-31,2019-12-31,42,0.004682,0.014355,1.865814,0.0621,41,3
4,vw,2,1,2020-01-31,2020-06-30,6,0.016401,0.018196,3.409246,0.0007,5,2
5,vw,3,0,2020-07-31,2025-09-30,63,0.000470,0.018074,0.197722,0.8433,62,3


### 2.2 Predicted Returns Across ESG Momentum Payoffs

#### 2.2.1 Helper Functions

In [ ]:
def predict_returns_across_esg_momentum_portfolios(
    df,
    date_col= date_column,
    stock_col = stock_column,
    ret_col= "stock_return",

    # Macro columns (DIV, YLD, TERM, DEF)
    div_col= "dividend_yield_ttm",
    yld_col= "spot_rate_3m_government_bond",
    term_col= "term_spread",
    def_col= "default_spread",

    # Windows
    train_start_lag = 24,   # t-24
    train_end_lag = 1,     # t-1

    event_start_offset = -11, # First displayed month
    event_end_offset = 1,   # Holding Month

    min_train_obs = 24,     # safety threshold

):
    """
    For each stock and each formation month t:
    1) Estimate a stock-level macro prediction model from train_start_lag to train_end_lag (e.g., t-24 to t-1)
    2) Keep coefficients fixed
    3) Predict returns for event months 0 and 1
    4) Attach the stock's ESG-momentum portfolio membership from the formation month t
    """

    # --- Get monthly portfolios & Prepare Data for Regression ---

    data = df.copy()
    data[date_col] = pd.to_datetime(data[date_col])

    regressor_cols = [div_col, yld_col, term_col, def_col]

    # Create regressor variables by shifting the date column to align with the prediction window (t+1) and renaming the date column for clarity
    regressor_variables = data[[date_col, stock_col] + regressor_cols].copy()
    regressor_variables["predicted_date"] = regressor_variables[date_col] + pd.offsets.MonthEnd(+1)
    regressor_variables.rename(columns={date_col: "regressor_date"}, inplace=True)

    # Create regressed variable by selecting the date, stock, and return columns, and renaming the date column to align with the regressor variables
    regressed_variable = data[[date_col, stock_col, ret_col]].copy()
    regressed_variable = regressed_variable.rename(
        columns={
            date_col: "predicted_date", 
            ret_col: "realized_return"
        }
    )

    # Merge the regressed variable with the regressor variables on the predicted_date and stock columns to create a combined DataFrame for regression analysis
    merged = regressed_variable.merge(
        regressor_variables,
        on=["predicted_date", stock_col],
        how="inner"
    )

    # Reorder columns in the merged DataFrame for better readability, keeping the predicted_date, regressor_date, stock column, realized return, and then the regressor columns in a specific order
    merged = merged[
        ["predicted_date", "regressor_date", stock_col, "realized_return"] + regressor_cols
    ].sort_values([stock_col, "predicted_date"]).reset_index(drop=True)

    predictions_all = []

    # Calculate the minimum and maximum formation dates based on the predicted_date column in the merged DataFrame, 
    minimum_formation_date = merged["predicted_date"].min() + pd.offsets.MonthEnd(train_start_lag)
    maximum_formation_date = merged["predicted_date"].max() - pd.offsets.MonthEnd(event_end_offset)

    for stock, stock_data in merged.groupby(stock_col):
        stock_data = stock_data.sort_values("predicted_date").reset_index(drop=True)

        # Loop over formation months from minimum to maximum formation date, with a frequency of 1 month end
        for formation_date in pd.date_range(start=minimum_formation_date, end=maximum_formation_date, freq="ME"):

            # --- Estimate regression on t-67 through t-12 ---

            # Calculate the start and end dates for training period
            train_start_date = formation_date - pd.offsets.MonthEnd(train_start_lag)
            train_end_date = formation_date - pd.offsets.MonthEnd(train_end_lag)

            # Select training data
            training_data = stock_data[
                (stock_data["predicted_date"] >= train_start_date) &
                (stock_data["predicted_date"] <= train_end_date)
            ].copy()

            # Require a minimum estimation-window length.
            if len(training_data) < min_train_obs:
                continue
            
            # Add a constant term to the regressors for the OLS regression, ensuring that the constant is included in the model
            X_train = sm.add_constant(training_data[regressor_cols], has_constant="add")
            y_train = training_data["realized_return"]

            # Train the model
            model = sm.OLS(y_train, X_train).fit()

            # --- Predict in event months t-11 through t+1 ---

            # Obtain the dates for which to make predictions
            prediction_dates = [
                formation_date + pd.offsets.MonthEnd(k)
                for k in range(event_start_offset, event_end_offset + 1)
            ]

            # Subset data
            prediction_data = stock_data[
                stock_data["predicted_date"].isin(prediction_dates)
            ].copy()

            # If there is not prediction data continue
            if prediction_data.empty:
                continue

            # Make prediction
            X_predict = sm.add_constant(prediction_data[regressor_cols], has_constant="add")
            prediction_data["predicted_return"] = model.predict(X_predict)

            # Add formation date and training dates to the prediction data
            prediction_data["formation_date"] = formation_date
            prediction_data["training_start_date"] = train_start_date
            prediction_data["training_end_date"] = train_end_date

            # Compute event month from formation date (e.g., t-11, t-10, ..., t+1) and add it to the prediction data
            prediction_data["event_month"] = [
                (pred_date.year - formation_date.year) * 12 + (pred_date.month - formation_date.month)
                for pred_date in prediction_data["predicted_date"]
            ]

            # Keep only relevant columns for the output and append to the list of predictions
            prediction_data = prediction_data[
                [
                    "predicted_date",
                    stock_col,
                    "formation_date",
                    "event_month",
                    "training_start_date",
                    "training_end_date",
                    "predicted_return"
                ]
            ]

            # Append the prediction data for the current stock and formation month to the list of all predictions
            predictions_all.append(prediction_data)

    if not predictions_all:
        return pd.DataFrame(
            columns=[
                "predicted_date",
                stock_col,
                "formation_date",
                "event_month",
                "training_start_date",
                "training_end_date",
                "predicted_return",
                "portfolio"
            ]
        )

    # Concatenate all the prediction data into a single DataFrame, ignoring the index to create a continuous index for the combined data
    outpt_data = pd.concat(predictions_all, axis=0, ignore_index=True)

    # Get the portfolio assignment
    portfolios_data = data[[date_col, stock_col, "portfolio"]].copy()

    # Merge the output data with the portfolio data to attach the portfolio assignment
    outpt_data = outpt_data.merge(
        portfolios_data,
        left_on=["formation_date", stock_col],
        right_on=[date_col, stock_col],
        how="left"
    )

    # Drop not needed columns
    outpt_data = outpt_data.drop(columns=[date_col]).dropna(subset=["portfolio"]).copy()
    outpt_data["portfolio"] = outpt_data["portfolio"].astype(int)

    outpt_data = outpt_data.sort_values(
        ["formation_date", "event_month", "portfolio", stock_col]
    ).reset_index(drop=True)

    return outpt_data

In [ ]:
def compute_median_predicted_returns(predicted_returns):

    # --- Compute median predicted returns ---

    # Calculate median predicted return for each combination of formation date, event month, and portfolio
    median_predicted_return_by_event_month_and_date = (
        predicted_returns
        .groupby(["formation_date", "event_month", "portfolio"])["predicted_return"]
        .median()
        .reset_index()
    )
    
    # Corner portfolios: P1 and P5
    top_portfolio = median_predicted_return_by_event_month_and_date[
        median_predicted_return_by_event_month_and_date["portfolio"] == number_of_portfolios
    ].copy()

    bottom_portfolio = median_predicted_return_by_event_month_and_date[
        median_predicted_return_by_event_month_and_date["portfolio"] == 1
    ].copy()

    diff_df = top_portfolio.merge(
        bottom_portfolio,
        on=["formation_date", "event_month"],
        suffixes=(f"_{number_of_portfolios}", "_1")
    )

    # Calculate the difference in median predicted returns between the top portfolio and the bottom portfolio for each formation date and event month
    diff_df["difference"] = (
        diff_df[f"predicted_return_{number_of_portfolios}"] - diff_df["predicted_return_1"]
    )

    # Calculate the median predicted return for each event month and portfolio across all formation dates, then sort by event month and portfolio for better readability
    median_predicted_return_by_event_month = (
        predicted_returns
        .groupby(["portfolio", "event_month"])["predicted_return"]
        .median()
        .reset_index()
        .sort_values(["event_month", "portfolio"])
        .reset_index(drop=True)
    )

    return median_predicted_return_by_event_month, diff_df

In [57]:
def perform_sign_test(data, column, alternative="two-sided"):

    x = pd.Series(data[column]).dropna()

    # Count the number of times the difference is positive, negative, and zero
    positive_count = (x > 0).sum()
    negative_count = (x < 0).sum()
    zero_count = (x == 0).sum()

    # Perform the binomial test under the null that the probability of a positive difference is 0.5
    if (positive_count + negative_count) != 0:
        result = binomtest(
            positive_count,
            n=positive_count + negative_count,
            p=0.5,
            alternative=alternative
        )
    else:
        return {
            "non_zero": 0,
            "positive_count": positive_count,
            "negative_count": negative_count,
            "zero_count": zero_count,
            "p_value": np.nan,
        }

    return {
        "non_zero": positive_count + negative_count,
        "positive_count": positive_count,
        "negative_count": negative_count,
        "zero_count": zero_count,
        "p_value": result.pvalue,
    }


In [ ]:
def perform_Sign_and_T_Test(diff_df):

    results_sign_test = []
    alternatives = ["two-sided", "greater"]

    # Loop over each unique event month in the diff_df DataFrame, sorted in ascending order, and perform the sign test on the "difference" column for each event month, 
    # appending the results to the results_sign_test list as a dictionary containing the event month and the results of the sign test
    for event_month in sorted(diff_df["event_month"].dropna().unique()):
        for alternative in alternatives:
            results = perform_sign_test(
                diff_df[diff_df["event_month"] == event_month],
                "difference", alternative=alternative
            )

            results_sign_test.append({
                "event_month": int(event_month),
                "alternative": alternative,
                "average_difference": diff_df[diff_df["event_month"] == event_month]["difference"].mean(),
                "median_difference": diff_df[diff_df["event_month"] == event_month]["difference"].median(),
                **results
            })

    results_sign_test = (
        pd.DataFrame(results_sign_test)
        .sort_values("event_month")
        .reset_index(drop=True)
    )

    ttest_event_month_0 = calculate_newey_west_stats(diff_df[diff_df["event_month"] == 0]["difference"])
    ttest_event_month_1 = calculate_newey_west_stats(diff_df[diff_df["event_month"] == 1]["difference"])

    return results_sign_test, ttest_event_month_0, ttest_event_month_1

#### 2.2.2. Analysis

##### 36 Months

In [30]:
predicted_returns_across_esg_momentum_portfolios_36 = predict_returns_across_esg_momentum_portfolios(
    portfolios,
    train_start_lag=36,
    train_end_lag=1,
    event_start_offset=0,
    event_end_offset=1,
    min_train_obs=24,
    ret_col="stock_return",
)

In [31]:
median_predicted_return_by_event_month_36, diff_df_36 = compute_median_predicted_returns(predicted_returns_across_esg_momentum_portfolios_36)

In [32]:
results_sign_test_36, ttest_event_month_0_36, ttest_event_month_1_36 = perform_Sign_and_T_Test(diff_df_36)

/tmp/ipykernel_697/3769629738.py:21: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  "mean": model.params[0],
/tmp/ipykernel_697/3769629738.py:23: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  "t_statistic": model.tvalues[0],
/tmp/ipykernel_697/3769629738.py:24: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  "p_value": round(model.pvalues[0], 4),
/tmp/ipykernel_697/3769629738.py:21: FutureWarning: Series.__getitem__ treating keys as po

In [33]:
results_sign_test_36.head(10)

,event_month,alternative,average_difference,median_difference,non_zero,positive_count,negative_count,zero_count,p_value
0,0,two-sided,0.002038,0.001564,74,46,28,0,0.047393
1,0,greater,0.002038,0.001564,74,46,28,0,0.023696
2,1,two-sided,0.002948,0.002562,74,44,30,0,0.130178
3,1,greater,0.002948,0.002562,74,44,30,0,0.065089


In [34]:
print("T-test results for event month 0 (formation month):")
print(ttest_event_month_0_36)
print("")
print("T-test results for event month 1 (holding month):")
print(ttest_event_month_1_36)

T-test results for event month 0 (formation month):
       mean   std_dev  t_statistic  p_value  degrees_of_freedom   n  max_lags
0  0.002038  0.009629     1.347274   0.1779                  73  74         3

T-test results for event month 1 (holding month):
       mean   std_dev  t_statistic  p_value  degrees_of_freedom   n  max_lags
0  0.002948  0.011082     1.743049   0.0813                  73  74         3


In [35]:
median_predicted_return_by_event_month_36

,portfolio,event_month,predicted_return
0,1,0,0.011475
1,2,0,0.010491
2,3,0,0.010731
3,4,0,0.011037
4,5,0,0.013754
5,1,1,0.011697
6,2,1,0.010502
7,3,1,0.010589
8,4,1,0.011480
9,5,1,0.014371


##### 48 Months

In [37]:
predicted_returns_across_esg_momentum_portfolios_48 = predict_returns_across_esg_momentum_portfolios(
    portfolios,
    train_start_lag=48,
    train_end_lag=1,
    event_start_offset=0,
    event_end_offset=1,
    min_train_obs=24,
    ret_col="stock_return",
)

In [38]:
median_predicted_return_by_event_month_48, diff_df_48 = compute_median_predicted_returns(predicted_returns_across_esg_momentum_portfolios_48)

In [39]:
results_sign_test_48, ttest_event_month_0_48, ttest_event_month_1_48 = perform_Sign_and_T_Test(diff_df_48)

/tmp/ipykernel_697/3769629738.py:21: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  "mean": model.params[0],
/tmp/ipykernel_697/3769629738.py:23: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  "t_statistic": model.tvalues[0],
/tmp/ipykernel_697/3769629738.py:24: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  "p_value": round(model.pvalues[0], 4),
/tmp/ipykernel_697/3769629738.py:21: FutureWarning: Series.__getitem__ treating keys as po

In [40]:
results_sign_test_48.head(10)

,event_month,alternative,average_difference,median_difference,non_zero,positive_count,negative_count,zero_count,p_value
0,0,two-sided,0.002059,0.001341,62,40,22,0,0.030016
1,0,greater,0.002059,0.001341,62,40,22,0,0.015008
2,1,two-sided,0.002890,0.001481,62,36,26,0,0.252854
3,1,greater,0.002890,0.001481,62,36,26,0,0.126427


In [41]:
print("T-test results for event month 0 (formation month):")
print(ttest_event_month_0_48)
print("")
print("T-test results for event month 1 (holding month):")
print(ttest_event_month_1_48)

T-test results for event month 0 (formation month):
       mean   std_dev  t_statistic  p_value  degrees_of_freedom   n  max_lags
0  0.002059  0.009701     1.266663   0.2053                  61  62         3

T-test results for event month 1 (holding month):
      mean   std_dev  t_statistic  p_value  degrees_of_freedom   n  max_lags
0  0.00289  0.010451     1.491776   0.1358                  61  62         3


In [42]:
median_predicted_return_by_event_month_48

,portfolio,event_month,predicted_return
0,1,0,0.001907
1,2,0,0.001294
2,3,0,0.000744
3,4,0,-0.000087
4,5,0,0.003777
5,1,1,0.000652
6,2,1,-0.000109
7,3,1,-0.001296
8,4,1,-0.001486
9,5,1,0.002884


##### 60 Months

In [44]:
predicted_returns_across_esg_momentum_portfolios_60 = predict_returns_across_esg_momentum_portfolios(
    portfolios,
    train_start_lag=60,
    train_end_lag=1,
    event_start_offset=0,
    event_end_offset=1,
    min_train_obs=24,
    ret_col="stock_return",
)

In [45]:
median_predicted_return_by_event_month_60, diff_df_60 = compute_median_predicted_returns(predicted_returns_across_esg_momentum_portfolios_60)

In [46]:
results_sign_test_60, ttest_event_month_0_60, ttest_event_month_1_60 = perform_Sign_and_T_Test(diff_df_60)

/tmp/ipykernel_697/3769629738.py:21: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  "mean": model.params[0],
/tmp/ipykernel_697/3769629738.py:23: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  "t_statistic": model.tvalues[0],
/tmp/ipykernel_697/3769629738.py:24: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  "p_value": round(model.pvalues[0], 4),
/tmp/ipykernel_697/3769629738.py:21: FutureWarning: Series.__getitem__ treating keys as po

In [47]:
results_sign_test_60.head(10)

,event_month,alternative,average_difference,median_difference,non_zero,positive_count,negative_count,zero_count,p_value
0,0,two-sided,0.004796,0.002614,50,38,12,0,0.000306
1,0,greater,0.004796,0.002614,50,38,12,0,0.000153
2,1,two-sided,0.005109,0.002747,50,40,10,0,0.000024
3,1,greater,0.005109,0.002747,50,40,10,0,0.000012


In [48]:
print("T-test results for event month 0 (formation month):")
print(ttest_event_month_0_60)
print("")
print("T-test results for event month 1 (holding month):")
print(ttest_event_month_1_60)

T-test results for event month 0 (formation month):
       mean   std_dev  t_statistic  p_value  degrees_of_freedom   n  max_lags
0  0.004796  0.011242     2.322317   0.0202                  49  50         3

T-test results for event month 1 (holding month):
       mean   std_dev  t_statistic  p_value  degrees_of_freedom   n  max_lags
0  0.005109  0.009648     2.474248   0.0134                  49  50         3


In [49]:
median_predicted_return_by_event_month_60

,portfolio,event_month,predicted_return
0,1,0,0.000695
1,2,0,0.001178
2,3,0,-0.001237
3,4,0,-0.001339
4,5,0,0.003228
5,1,1,-0.000014
6,2,1,0.000304
7,3,1,-0.002285
8,4,1,-0.002335
9,5,1,0.003046


### 2.3 ESG Momentum Payoffs After Adjusting for Predicted Returns

#### 2.3.1 Helper Functions

In [54]:
def estimate_stock_level_adjusted_returns(
    df,
    date_col = date_column,
    stock_col = stock_column,
    include_intercept = False,
    ret_col = "stock_return",
    div_col = "dividend_yield_ttm",
    yld_col = "spot_rate_3m_government_bond",
    term_col = "term_spread",
    def_col = "default_spread",
    window_months = 60,
    min_train_obs = 24,
):
    """
    For each stock and each holding month h:

    1) Estimate stock return on lagged macro variables using months h-W through h-1
       (calendar window; requires at least min_train_obs usable rows).
    2) Predict only the slope-based macro component for month h.
    3) Compute:
           adjusted_return_h = actual_return_h - predicted_macro_component_h
                             = intercept_h + residual_h
    """

    data = df.copy()
    data[date_col] = pd.to_datetime(data[date_col])

    regressor_cols = [div_col, yld_col, term_col, def_col]

    # Regressors dated s-1, used to predict return in month s
    regressor_variables = data[[date_col, stock_col] + regressor_cols].copy()
    regressor_variables["predicted_date"] = regressor_variables[date_col] + pd.offsets.MonthEnd(1)
    regressor_variables = regressor_variables.rename(columns={date_col: "regressor_date"})

    # Realized return in month s
    regressed_variable = data[[date_col, stock_col, ret_col]].copy()
    regressed_variable = regressed_variable.rename(
        columns={
            date_col: "predicted_date",
            ret_col: "realized_return"
        }
    )

    # Align month-s return with month-(s-1) macro variables
    merged = regressed_variable.merge(
        regressor_variables,
        on=["predicted_date", stock_col],
        how="inner"
    )

    # Keep only rows with non-missing realized returns and regressor values, and reorder columns for better readability
    merged = (
        merged
        .dropna(subset=["realized_return"] + regressor_cols)
        [["predicted_date", "regressor_date", stock_col, "realized_return"] + regressor_cols]
        .sort_values([stock_col, "predicted_date"])
        .reset_index(drop=True)
    )

    # Calculate the first holding date for which a full lookback window exists, which is the minimum predicted_date plus the specified window size in months
    first_holding_date = merged["predicted_date"].min() + pd.offsets.MonthEnd(window_months)

    adjustments_all = []

    for stock, stock_data in merged.groupby(stock_col):

        stock_data = stock_data.sort_values("predicted_date").reset_index(drop=True)

        # Get all holding dates for which a full lookback window exists, i.e., all predicted_dates that are greater than or equal to the first_holding_date
        eligible_holding_dates = stock_data.loc[stock_data["predicted_date"] >= first_holding_date, "predicted_date"]

        # Loop over each eligible holding date to estimate the regression model and compute the adjusted return for that month
        for holding_date in eligible_holding_dates:

            # Calculate the start and end dates for the training period based on the current holding date and the specified window size
            train_start_date = holding_date - pd.offsets.MonthEnd(window_months)
            train_end_date = holding_date - pd.offsets.MonthEnd(1)

            training_data = stock_data[
                (stock_data["predicted_date"] >= train_start_date) &
                (stock_data["predicted_date"] <= train_end_date)
            ].copy()

            if len(training_data) < min_train_obs:
                continue

            # Add a constant term to the regressors for the OLS regression, ensuring that the constant is included in the model
            X_train = sm.add_constant(training_data[regressor_cols].astype(float), has_constant="add")
            y_train = training_data["realized_return"].astype(float)

            # Train the model
            model = sm.OLS(y_train, X_train).fit()

            # Get the regressor values for the current holding date to compute the predicted macro component.
            current_row = stock_data.loc[stock_data["predicted_date"] == holding_date].iloc[0]
            current_x = current_row[regressor_cols].astype(float)

            # Predicted macro component.
            # If intercept is included in macro component, it is subtracted from the realized return, which means that adjusted return only 
            # keeps the residual component.
            if include_intercept:
                predicted_macro_component = float(
                    np.dot(current_x.values, model.params[regressor_cols].values) + model.params["const"]
                )
            else: 
                predicted_macro_component = float(
                    np.dot(current_x.values, model.params[regressor_cols].values)
                )

            # Compute the intercept, fitted return, residual, and adjusted return for the current holding date.
            intercept = float(model.params["const"])

            # If the intercept is included in the macro component, then the fitted return is just the predicted macro component.
            if include_intercept:
                fitted_return = predicted_macro_component
            else:
                fitted_return = intercept + predicted_macro_component

            realized_return = float(current_row["realized_return"])
            residual = realized_return - fitted_return

            # Keep intercept + residual; Removes the slope based macro component
            adjusted_return = realized_return - predicted_macro_component

            adjustments_all.append({
                date_col: holding_date,
                stock_col: stock,
                "regressor_date": current_row["regressor_date"],
                "realized_return": realized_return,
                "predicted_macro_component": predicted_macro_component,
                "intercept": intercept,
                "fitted_return": fitted_return,
                "residual": residual,
                "adjusted_return": adjusted_return,
                "window_start": train_start_date,
                "window_end": train_end_date,
                "n_obs": len(training_data),
                "return_used_in_regression": ret_col,
            })

    adjusted_returns = pd.DataFrame(adjustments_all)

    # Merge adjusted returns back onto the original stock-month panel
    data_with_adjusted_returns = data.merge(
        adjusted_returns[
            [date_col, stock_col, "realized_return", "adjusted_return", "predicted_macro_component", "intercept", "residual"]
        ],
        on=[date_col, stock_col],
        how="left"
    )

    return adjusted_returns, data_with_adjusted_returns

In [59]:
def analyze_esg_momentum_payoffs_after_adjusting_for_predicted_returns(
    df,
    date_col = date_column,
    stock_col = stock_column,
    portfolio_col = "portfolio",
    mcap_col = "mcap",
    n_portfolios = number_of_portfolios,
    include_intercept = False,
    ret_col = "stock_return",
    div_col = "dividend_yield_ttm",
    yld_col = "spot_rate_3m_government_bond",
    term_col = "term_spread",
    def_col = "default_spread",
    window_months = 60,
    min_train_obs = 24,
):
    """
    Full ESG momentum payoff analysis after adjusting for predicted returns.

    Steps:
    1) Estimate stock-level adjusted returns in holding month h = t+1.
    2) Compute adjusted portfolio returns using portfolios formed in month t.
    3) Compute adjusted ESG-momentum payoffs: P5 - P1.
    4) Also compute raw payoffs on the same sample dates for comparison.
    5) Report t-test and sign-test results.
    """

    # 1) Stock-level adjusted returns
    adjusted_stock_returns, df_with_adjusted_returns = estimate_stock_level_adjusted_returns(
        df=df,
        date_col=date_col,
        stock_col=stock_col,
        include_intercept=include_intercept,
        ret_col=ret_col,
        div_col=div_col,
        yld_col=yld_col,
        term_col=term_col,
        def_col=def_col,
        window_months=window_months,
        min_train_obs=min_train_obs,
    )

    # 2) Adjusted next-month portfolio returns
    adjusted_portfolio_returns = compute_next_month_portfolio_returns(
        df_with_portfolios=df_with_adjusted_returns,
        date_col=date_col,
        stock_col=stock_col,
        ret_col="adjusted_return",
        mcap_col=mcap_col,
        portfolio_col=portfolio_col,
        n_portfolios=n_portfolios,
        month_end_shift=1
    )

    # Raw benchmark on the same return definition used in the regression
    raw_portfolio_returns = compute_next_month_portfolio_returns(
        df_with_portfolios=df_with_adjusted_returns,
        date_col=date_col,
        stock_col=stock_col,
        ret_col="realized_return",
        mcap_col=mcap_col,
        portfolio_col=portfolio_col,
        n_portfolios=n_portfolios,
        month_end_shift=1
    )

    # 3) Compute P5 - P1 payoffs
    adjusted_payoffs = adjusted_portfolio_returns.copy()
    raw_payoffs_same_sample = raw_portfolio_returns.copy()

    for weight_type in ["ew", "vw"]:
        top_col = f"{weight_type}_portfolio_{n_portfolios}"
        bottom_col = f"{weight_type}_portfolio_1"

        adjusted_payoffs[f"{weight_type}_esg_momentum_payoff_adjusted"] = (
            adjusted_payoffs[top_col] - adjusted_payoffs[bottom_col]
        )

        raw_payoffs_same_sample[f"{weight_type}_esg_momentum_payoff_raw"] = (
            raw_payoffs_same_sample[top_col] - raw_payoffs_same_sample[bottom_col]
        )
        
    # 4) Summary statistics
    summary_rows = []

    for weight_type in ["ew", "vw"]:
        adjusted_col = f"{weight_type}_esg_momentum_payoff_adjusted"
        raw_col = f"{weight_type}_esg_momentum_payoff_raw"

        adjusted_t = calculate_newey_west_stats(adjusted_payoffs[adjusted_col]).iloc[0].to_dict()
        adjusted_sign = perform_sign_test(adjusted_payoffs, adjusted_col)

        raw_t = calculate_newey_west_stats(raw_payoffs_same_sample[raw_col]).iloc[0].to_dict()
        raw_sign = perform_sign_test(raw_payoffs_same_sample, raw_col)

        summary_rows.append({
            "series": f"{weight_type.upper()} adjusted payoff",
            **adjusted_t,
            "sign_test_p_value": adjusted_sign["p_value"],
            "positive_count": adjusted_sign["positive_count"],
            "negative_count": adjusted_sign["negative_count"],
            "zero_count": adjusted_sign["zero_count"],
            "return_used_in_regression": ret_col,
            "window_months": window_months,
            "min_train_obs": min_train_obs,
        })

        summary_rows.append({
            "series": f"{weight_type.upper()} raw payoff (same sample)",
            **raw_t,
            "sign_test_p_value": raw_sign["p_value"],
            "positive_count": raw_sign["positive_count"],
            "negative_count": raw_sign["negative_count"],
            "zero_count": raw_sign["zero_count"],
            "return_used_in_regression": ret_col,
            "window_months": window_months,
            "min_train_obs": min_train_obs,
        })

    summary = pd.DataFrame(summary_rows)

    return {
        "stock_level_adjustments": adjusted_stock_returns,
        "data_with_adjusted_returns": df_with_adjusted_returns,
        "adjusted_portfolio_returns": adjusted_portfolio_returns,
        "adjusted_payoffs": adjusted_payoffs,
        "raw_payoffs_same_sample": raw_payoffs_same_sample,
        "summary": summary,
    }

#### 2.3.2 Analysis

##### 2.3.2.1 Adjusted Return Includes intercept

In [60]:
adjusted_payoff_results_36 = analyze_esg_momentum_payoffs_after_adjusting_for_predicted_returns(
    df=portfolios,
    window_months=36,
    min_train_obs=24,
)

/tmp/ipykernel_697/3793592047.py:45: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  long = merged.groupby(["holding_date", portfolio_col]).apply(_agg).reset_index()
/tmp/ipykernel_697/3793592047.py:45: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  long = merged.groupby(["holding_date", portfolio_col]).apply(_agg).reset_index()
/tmp/ipykernel_697/3769629738.py:21: FutureWarning: Series.__getitem__ treating keys as posi

In [61]:
adjusted_payoff_results_48 = analyze_esg_momentum_payoffs_after_adjusting_for_predicted_returns(
    df=portfolios, 
    window_months=48,
    min_train_obs=24,
)

/tmp/ipykernel_697/3793592047.py:45: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  long = merged.groupby(["holding_date", portfolio_col]).apply(_agg).reset_index()
/tmp/ipykernel_697/3793592047.py:45: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  long = merged.groupby(["holding_date", portfolio_col]).apply(_agg).reset_index()
/tmp/ipykernel_697/3769629738.py:21: FutureWarning: Series.__getitem__ treating keys as posi

In [62]:
adjusted_payoff_results_60 = analyze_esg_momentum_payoffs_after_adjusting_for_predicted_returns(
    df=portfolios,
    window_months=60,
    min_train_obs=24,
)

/tmp/ipykernel_697/3793592047.py:45: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  long = merged.groupby(["holding_date", portfolio_col]).apply(_agg).reset_index()
/tmp/ipykernel_697/3793592047.py:45: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  long = merged.groupby(["holding_date", portfolio_col]).apply(_agg).reset_index()
/tmp/ipykernel_697/3769629738.py:21: FutureWarning: Series.__getitem__ treating keys as posi

In [63]:
adjusted_payoff_results_36["summary"].head()

,series,mean,std_dev,t_statistic,p_value,degrees_of_freedom,n,max_lags,sign_test_p_value,positive_count,negative_count,zero_count,return_used_in_regression,window_months,min_train_obs
0,EW adjusted payoff,0.012179,0.035962,1.753261,0.0796,74.0,75.0,3.0,0.355699,42,33,0,stock_return,36,24
1,EW raw payoff (same sample),0.000962,0.012814,0.642317,0.5207,74.0,75.0,3.0,0.165428,44,31,0,stock_return,36,24
2,VW adjusted payoff,-0.016567,0.037573,-2.541518,0.0110,74.0,75.0,3.0,0.002444,24,51,0,stock_return,36,24
3,VW raw payoff (same sample),0.001422,0.017084,0.706655,0.4798,74.0,75.0,3.0,0.488683,41,34,0,stock_return,36,24


In [64]:
adjusted_payoff_results_48["summary"].head()

,series,mean,std_dev,t_statistic,p_value,degrees_of_freedom,n,max_lags,sign_test_p_value,positive_count,negative_count,zero_count,return_used_in_regression,window_months,min_train_obs
0,EW adjusted payoff,0.003544,0.027095,0.651754,0.5146,62.0,63.0,3.0,0.801306,33,30,0,stock_return,48,24
1,EW raw payoff (same sample),0.001439,0.013031,0.851139,0.3947,62.0,63.0,3.0,0.207368,37,26,0,stock_return,48,24
2,VW adjusted payoff,-0.016913,0.034304,-2.567871,0.0102,62.0,63.0,3.0,0.000898,18,45,0,stock_return,48,24
3,VW raw payoff (same sample),0.000771,0.017882,0.338317,0.7351,62.0,63.0,3.0,0.614655,34,29,0,stock_return,48,24


In [65]:
adjusted_payoff_results_60["summary"].head()

,series,mean,std_dev,t_statistic,p_value,degrees_of_freedom,n,max_lags,sign_test_p_value,positive_count,negative_count,zero_count,return_used_in_regression,window_months,min_train_obs
0,EW adjusted payoff,-0.002340,0.022341,-0.493880,0.6214,50.0,51.0,3.0,1.000000,25,26,0,stock_return,60,24
1,EW raw payoff (same sample),0.001876,0.011188,1.048979,0.2942,50.0,51.0,3.0,0.262438,30,21,0,stock_return,60,24
2,VW adjusted payoff,-0.018997,0.030191,-3.083230,0.0020,50.0,51.0,3.0,0.000057,11,40,0,stock_return,60,24
3,VW raw payoff (same sample),0.000478,0.016297,0.189307,0.8499,50.0,51.0,3.0,1.000000,26,25,0,stock_return,60,24


##### 2.3.2.2 Adjusted Payoffs Excludes Intercept

In [66]:
adjusted_payoff_results_36_exc_int = analyze_esg_momentum_payoffs_after_adjusting_for_predicted_returns(
    df=portfolios,
    include_intercept=True, # Include in Macro Predicted Component => Excludes from Adjusted Return
    window_months=36,
    min_train_obs=24,
)

/tmp/ipykernel_697/3793592047.py:45: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  long = merged.groupby(["holding_date", portfolio_col]).apply(_agg).reset_index()
/tmp/ipykernel_697/3793592047.py:45: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  long = merged.groupby(["holding_date", portfolio_col]).apply(_agg).reset_index()
/tmp/ipykernel_697/3769629738.py:21: FutureWarning: Series.__getitem__ treating keys as posi

In [67]:
adjusted_payoff_results_48_exc_int = analyze_esg_momentum_payoffs_after_adjusting_for_predicted_returns(
    df=portfolios,
    include_intercept=True,
    window_months=48,
    min_train_obs=24,
)

/tmp/ipykernel_697/3793592047.py:45: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  long = merged.groupby(["holding_date", portfolio_col]).apply(_agg).reset_index()
/tmp/ipykernel_697/3793592047.py:45: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  long = merged.groupby(["holding_date", portfolio_col]).apply(_agg).reset_index()
/tmp/ipykernel_697/3769629738.py:21: FutureWarning: Series.__getitem__ treating keys as posi

In [68]:
adjusted_payoff_results_60_exc_int = analyze_esg_momentum_payoffs_after_adjusting_for_predicted_returns(
    df=portfolios,
    include_intercept=True,
    window_months=60,
    min_train_obs=24,
)

/tmp/ipykernel_697/3793592047.py:45: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  long = merged.groupby(["holding_date", portfolio_col]).apply(_agg).reset_index()
/tmp/ipykernel_697/3793592047.py:45: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  long = merged.groupby(["holding_date", portfolio_col]).apply(_agg).reset_index()
/tmp/ipykernel_697/3769629738.py:21: FutureWarning: Series.__getitem__ treating keys as posi

In [69]:
adjusted_payoff_results_36_exc_int["summary"].head()

,series,mean,std_dev,t_statistic,p_value,degrees_of_freedom,n,max_lags,sign_test_p_value,positive_count,negative_count,zero_count,return_used_in_regression,window_months,min_train_obs
0,EW adjusted payoff,-0.000643,0.015447,-0.352059,0.7248,74.0,75.0,3.0,0.817554,36,39,0,stock_return,36,24
1,EW raw payoff (same sample),0.000962,0.012814,0.642317,0.5207,74.0,75.0,3.0,0.165428,44,31,0,stock_return,36,24
2,VW adjusted payoff,-0.001219,0.020794,-0.591151,0.5544,74.0,75.0,3.0,1.000000,37,38,0,stock_return,36,24
3,VW raw payoff (same sample),0.001422,0.017084,0.706655,0.4798,74.0,75.0,3.0,0.488683,41,34,0,stock_return,36,24


In [70]:
adjusted_payoff_results_48_exc_int["summary"].head()

,series,mean,std_dev,t_statistic,p_value,degrees_of_freedom,n,max_lags,sign_test_p_value,positive_count,negative_count,zero_count,return_used_in_regression,window_months,min_train_obs
0,EW adjusted payoff,0.000713,0.014405,0.400992,0.6884,62.0,63.0,3.0,0.313503,36,27,0,stock_return,48,24
1,EW raw payoff (same sample),0.001439,0.013031,0.851139,0.3947,62.0,63.0,3.0,0.207368,37,26,0,stock_return,48,24
2,VW adjusted payoff,0.001123,0.020544,0.486684,0.6265,62.0,63.0,3.0,0.207368,37,26,0,stock_return,48,24
3,VW raw payoff (same sample),0.000771,0.017882,0.338317,0.7351,62.0,63.0,3.0,0.614655,34,29,0,stock_return,48,24


In [71]:
adjusted_payoff_results_60_exc_int["summary"].head()

,series,mean,std_dev,t_statistic,p_value,degrees_of_freedom,n,max_lags,sign_test_p_value,positive_count,negative_count,zero_count,return_used_in_regression,window_months,min_train_obs
0,EW adjusted payoff,0.000116,0.013036,0.056345,0.9551,50.0,51.0,3.0,1.000000,26,25,0,stock_return,60,24
1,EW raw payoff (same sample),0.001876,0.011188,1.048979,0.2942,50.0,51.0,3.0,0.262438,30,21,0,stock_return,60,24
2,VW adjusted payoff,-0.000450,0.018884,-0.168480,0.8662,50.0,51.0,3.0,1.000000,26,25,0,stock_return,60,24
3,VW raw payoff (same sample),0.000478,0.016297,0.189307,0.8499,50.0,51.0,3.0,1.000000,26,25,0,stock_return,60,24


### 2.4 ESG Momentum Payoffs Regressed on Macroeconomic Variables

#### 2.4.1 Preparing Data

In [74]:
esg_momentum_payoff = portfolio_returns.copy()

for weight_type in ["ew", "vw"]:
    top_portfolio_col = f"{weight_type}_portfolio_{number_of_portfolios}"
    portfolio_1_col = f"{weight_type}_portfolio_1"

    # Calculate the esg momentum payoff as the difference between the returns of portfolio 5 and portfolio 1 for the current weight type
    esg_momentum_payoff[f"{weight_type}_esg_momentum_payoff"] = esg_momentum_payoff[top_portfolio_col] - esg_momentum_payoff[portfolio_1_col]

esg_momentum_payoff = esg_momentum_payoff[["date", "ew_esg_momentum_payoff", "vw_esg_momentum_payoff"]]
esg_momentum_payoff.dropna(inplace=True)

esg_momentum_payoff.to_csv(path + "07_esg_momentum_payoff.csv", index=False)

esg_momentum_payoff.head(5)

,date,ew_esg_momentum_payoff,vw_esg_momentum_payoff
0,2016-07-31,-0.007957,-0.014673
1,2016-08-31,0.003567,-0.017135
2,2016-09-30,-0.011599,-0.026532
3,2016-10-31,-0.026248,0.003359
4,2016-11-30,0.021207,0.020616


In [73]:
# Keep only the relevant columns for the regression analysis and drop duplicates to get unique combinations of date and macro variables
lagged_macro_variables = bc_data[["date", "dividend_yield_ttm", "spot_rate_3m_government_bond", "term_spread", "default_spread"]].drop_duplicates().reset_index(drop=True).copy()

# Offset date to align with the prediction window (t+1) and convert to date format
lagged_macro_variables["date"] = pd.to_datetime(lagged_macro_variables["date"]) + pd.offsets.MonthEnd(1)
lagged_macro_variables["date"] = lagged_macro_variables["date"].dt.date

lagged_macro_variables.head()

,date,dividend_yield_ttm,spot_rate_3m_government_bond,term_spread,default_spread
0,2016-07-31,0.045862,-0.006500,0.005456,0.0103
1,2016-08-31,0.044046,-0.006450,0.004932,0.0094
2,2016-09-30,0.043544,-0.006544,0.005367,0.0092
3,2016-10-31,0.043082,-0.007448,0.005828,0.0090
4,2016-11-30,0.043711,-0.008206,0.009636,0.0087


In [75]:
# Merge payoffs with macro variables
esg_momentum_payoff_with_macro = esg_momentum_payoff.merge(lagged_macro_variables, left_on="date", right_on="date", how="left")
esg_momentum_payoff_with_macro["date"] = pd.to_datetime(esg_momentum_payoff_with_macro["date"]).dt.date

esg_momentum_payoff_with_macro.to_csv(path + "07_esg_momentum_payoff_with_macro_variables.csv", index=False)

esg_momentum_payoff_with_macro.head()

,date,ew_esg_momentum_payoff,vw_esg_momentum_payoff,dividend_yield_ttm,spot_rate_3m_government_bond,term_spread,default_spread
0,2016-07-31,-0.007957,-0.014673,0.045862,-0.006500,0.005456,0.0103
1,2016-08-31,0.003567,-0.017135,0.044046,-0.006450,0.004932,0.0094
2,2016-09-30,-0.011599,-0.026532,0.043544,-0.006544,0.005367,0.0092
3,2016-10-31,-0.026248,0.003359,0.043082,-0.007448,0.005828,0.0090
4,2016-11-30,0.021207,0.020616,0.043711,-0.008206,0.009636,0.0087


#### 2.4.2 Helper Functions

In [ ]:
def regress_esg_momentum_on_macro(
    df,
    date_col = "date",
    payoff_col = "ew_esg_momentum_payoff",
    macro_cols = ["dividend_yield_ttm", "spot_rate_3m_government_bond", "term_spread", "default_spread"],
    window_months = 48
):
    """
        Implements a {window_months}-month rolling window.

        For each month t:
        1. Estimate payoff_s on macro_{s-1} using s = t-{window_months}, ..., t-1
        2. Predict payoff_t using macro_{t-1}
        3. Compute:
                residual_t  = payoff_t - fitted_t
                RES_t       = intercept_t + residual_t
    """

    # Prepare the regression dataset by selecting the relevant columns and dropping rows with missing values
    regression_data = df[[date_col, payoff_col] + macro_cols].dropna().copy()

    results = []

    # Loop over the regression dataset using a rolling window approach to estimate the regression model
    for i in range(window_months, len(regression_data)):

        estimation_window = regression_data.iloc[i - window_months:i].copy()
        current_row = regression_data.iloc[i].copy()

        y_train = estimation_window[payoff_col].astype(float)

        # The macro variables are already lagged by design, so they can directly be used for the regression without additional shifting
        X_train = sm.add_constant(estimation_window[macro_cols].astype(float), has_constant="add")

        # Fit the OLS regression 
        model = sm.OLS(y_train, X_train).fit()

        # Get the actual payoff at time t and the estimated coefficients from the regression model to compute the predicted payoff, residual, and RES_t for the current month t
        payoff_t = float(current_row[payoff_col])
        intercept_t = float(model.params["const"])

        beta_div_t = float(model.params["dividend_yield_ttm"])
        beta_yld_t = float(model.params["spot_rate_3m_government_bond"])
        beta_term_t = float(model.params["term_spread"])
        beta_def_t = float(model.params["default_spread"])

        div_t_minus_1 = float(current_row["dividend_yield_ttm"])
        yld_t_minus_1 = float(current_row["spot_rate_3m_government_bond"])
        term_t_minus_1 = float(current_row["term_spread"])
        def_t_minus_1 = float(current_row["default_spread"])

        predicted_macro_component_t = (
            beta_div_t * div_t_minus_1 +
            beta_yld_t * yld_t_minus_1 +
            beta_term_t * term_t_minus_1 +
            beta_def_t * def_t_minus_1
        )

        fitted_payoff_t = intercept_t + predicted_macro_component_t
        residual_t = payoff_t - fitted_payoff_t

        res_t = intercept_t + residual_t

        results.append({
            "date": current_row[date_col],
            "payoff_col": payoff_col,
            "actual_payoff_t": payoff_t,
            "predicted_payoff_t": fitted_payoff_t,
            "predicted_macro_component_t": predicted_macro_component_t,
            "residual_t": residual_t,
            "adj_r_squared": model.rsquared_adj,
            "RES_t": res_t,
            "INT_t": intercept_t,
            "DIV_beta_t": beta_div_t,
            "YLD_beta_t": beta_yld_t,
            "TERM_beta_t": beta_term_t,
            "DEF_beta_t": beta_def_t,
            "window_start": estimation_window[date_col].iloc[0],
            "window_end": estimation_window[date_col].iloc[-1],
            "n_obs": len(estimation_window)
        })
    
    results_df = pd.DataFrame(results)
    return results_df

In [77]:
def newey_west_mean_test(series, maxlags=None):
    """
    Tests whether the mean of a time series is different from zero
    using an intercept-only regression with HAC/Newey-West standard errors.
    """
    y = pd.Series(series).dropna().astype(float).reset_index(drop=True)
    n = len(y)

    if n <= 1:
        return {"mean": np.nan, "t_stat": np.nan, "p_value": np.nan, "n_obs": n, "max_lags": None}

    if maxlags is None:
        maxlags = int(np.floor(4 * (n / 100) ** (2 / 9)))
        
    effective_lags = min(maxlags, n - 1)

    X = np.ones((n, 1))
    model = sm.OLS(y, X).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": effective_lags}
    )

    return {
        "mean": float(model.params.iloc[0]),
        "t_stat": float(model.tvalues.iloc[0]),
        "p_value": float(model.pvalues.iloc[0]),
        "n_obs": int(len(y)),
        "max_lags": effective_lags
        
    }

In [78]:
def summarize_rolling_results(results_df):
    """
    Returns a Table-IV-style summary:
    average RES, INT, and average coefficients with Newey-West t-stats.
    Also reports % RES > 0 and sign-test p-value.
    """
    summary_targets = {
        "RES": "RES_t",
        "INT": "INT_t",
        "DIV": "DIV_beta_t",
        "YLD": "YLD_beta_t",
        "TERM": "TERM_beta_t",
        "DEF": "DEF_beta_t",
        "Adj_R_SQUARED": "adj_r_squared"
    }

    rows = []
    for label, col in summary_targets.items():
        stat = newey_west_mean_test(results_df[col])
        rows.append({
            "parameter": label,
            "mean": stat["mean"],
            "t_stat_NW": stat["t_stat"],
            "p_value_NW": round(stat["p_value"], 6),
            "n_obs": stat["n_obs"],
            "max_lags": stat["max_lags"]
        })

    # % RES > 0 and sign test
    res_series = results_df["RES_t"].dropna()
    n_positive = int((res_series > 0).sum())
    n_total = int(len(res_series))
    pct_positive = 100 * n_positive / n_total if n_total > 0 else np.nan
    sign_test_p = binomtest(n_positive, n=n_total, p=0.5, alternative="two-sided").pvalue if n_total > 0 else np.nan

    rows.append({
        "parameter": "% RES > 0",
        "mean": pct_positive,
        "t_stat_NW": np.nan,
        "p_value_NW": round(sign_test_p, 6),
        "n_obs": n_total,
        "max_lags": np.nan
    })

    return pd.DataFrame(rows)

#### 2.4.3 ESG Momentum Equal Weighted 

In [79]:
ew_entire_period_results_36 = regress_esg_momentum_on_macro(esg_momentum_payoff_with_macro, payoff_col="ew_esg_momentum_payoff", window_months=36)
ew_entire_period_results_48 = regress_esg_momentum_on_macro(esg_momentum_payoff_with_macro, payoff_col="ew_esg_momentum_payoff", window_months=48)
ew_entire_period_results_60 = regress_esg_momentum_on_macro(esg_momentum_payoff_with_macro, payoff_col="ew_esg_momentum_payoff", window_months=60)

ew_entire_period_table_36 = summarize_rolling_results(ew_entire_period_results_36)
ew_entire_period_table_48 = summarize_rolling_results(ew_entire_period_results_48)
ew_entire_period_table_60 = summarize_rolling_results(ew_entire_period_results_60)

In [80]:
ew_entire_period_results_36

,date,payoff_col,actual_payoff_t,predicted_payoff_t,predicted_macro_component_t,residual_t,adj_r_squared,RES_t,INT_t,DIV_beta_t,YLD_beta_t,TERM_beta_t,DEF_beta_t,window_start,window_end,n_obs
0,2019-07-31,ew_esg_momentum_payoff,0.009694,0.014614,-0.039206,-0.004920,-0.000898,0.048900,0.053820,-2.135376,-2.642266,-1.514219,3.379900,2016-07-31,2019-06-30,36
1,2019-08-31,ew_esg_momentum_payoff,0.005140,0.013633,-0.029151,-0.008493,-0.006515,0.034291,0.042784,-1.731204,-2.089393,-1.269967,3.020526,2016-08-31,2019-07-31,36
2,2019-09-30,ew_esg_momentum_payoff,-0.022316,0.010069,-0.025557,-0.032385,-0.016298,0.003241,0.035626,-1.581936,-1.671840,-0.884820,3.042963,2016-09-30,2019-08-31,36
3,2019-10-31,ew_esg_momentum_payoff,0.001466,0.001093,-0.015565,0.000372,-0.036417,0.017031,0.016658,-0.555680,1.827107,0.264239,2.132758,2016-10-31,2019-09-30,36
4,2019-11-30,ew_esg_momentum_payoff,-0.004475,0.005269,0.003683,-0.009744,-0.038056,-0.008158,0.001586,0.315131,2.373024,0.059615,0.767897,2016-11-30,2019-10-31,36
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,2025-05-31,ew_esg_momentum_payoff,0.005078,0.002632,0.091920,0.002446,0.207025,-0.086841,-0.089288,3.403519,-0.116978,-0.289834,-2.513043,2022-05-31,2025-04-30,36
71,2025-06-30,ew_esg_momentum_payoff,0.005299,0.002271,0.076114,0.003028,0.142253,-0.070815,-0.073843,3.015698,-0.200456,-0.334248,-2.488220,2022-06-30,2025-05-31,36
72,2025-07-31,ew_esg_momentum_payoff,0.000516,0.004697,0.067706,-0.004181,0.162823,-0.067190,-0.063008,2.880255,-0.404683,-0.488278,-2.577561,2022-07-31,2025-06-30,36
73,2025-08-31,ew_esg_momentum_payoff,-0.011081,0.003985,0.061905,-0.015065,0.120939,-0.072986,-0.057921,2.590448,-0.301757,-0.441499,-2.358345,2022-08-31,2025-07-31,36


In [81]:
ew_entire_period_table_36.head(10)

,parameter,mean,t_stat_NW,p_value_NW,n_obs,max_lags
0,RES,-0.003295,-0.292139,0.770180,75,3.0
1,INT,-0.000423,-0.037000,0.970485,75,3.0
2,DIV,1.031495,3.468717,0.000523,75,3.0
3,YLD,3.657487,3.415987,0.000636,75,3.0
4,TERM,-0.088094,-0.976223,0.328954,75,3.0
5,DEF,-0.772176,-1.919418,0.054931,75,3.0
6,Adj_R_SQUARED,0.102841,3.415795,0.000636,75,3.0
7,% RES > 0,52.000000,NaN,0.817554,75,NaN


In [82]:
ew_entire_period_table_48.head(10)

,parameter,mean,t_stat_NW,p_value_NW,n_obs,max_lags
0,RES,-0.010656,-1.264113,0.206190,63,3.0
1,INT,-0.008389,-0.960187,0.336961,63,3.0
2,DIV,1.059435,7.267919,0.000000,63,3.0
3,YLD,2.838974,3.123424,0.001788,63,3.0
4,TERM,-0.110318,-1.757460,0.078839,63,3.0
5,DEF,-0.731563,-3.281095,0.001034,63,3.0
6,Adj_R_SQUARED,0.067093,3.283437,0.001025,63,3.0
7,% RES > 0,38.095238,NaN,0.076926,63,NaN


In [83]:
ew_entire_period_table_60.head(10)

,parameter,mean,t_stat_NW,p_value_NW,n_obs,max_lags
0,RES,-0.015784,-2.713723,0.006653,51,3.0
1,INT,-0.013164,-1.920298,0.054820,51,3.0
2,DIV,0.886012,11.075663,0.000000,51,3.0
3,YLD,1.704754,2.303165,0.021270,51,3.0
4,TERM,-0.114854,-2.852782,0.004334,51,3.0
5,DEF,-0.356520,-2.028499,0.042509,51,3.0
6,Adj_R_SQUARED,0.044714,4.395119,0.000011,51,3.0
7,% RES > 0,27.450980,NaN,0.001769,51,NaN


#### 2.4.4 ESG Momentum Value Weighted

In [84]:
vw_entire_period_results_36 = regress_esg_momentum_on_macro(esg_momentum_payoff_with_macro, payoff_col="vw_esg_momentum_payoff", window_months=36)
vw_entire_period_results_48 = regress_esg_momentum_on_macro(esg_momentum_payoff_with_macro, payoff_col="vw_esg_momentum_payoff", window_months=48)
vw_entire_period_results_60 = regress_esg_momentum_on_macro(esg_momentum_payoff_with_macro, payoff_col="vw_esg_momentum_payoff", window_months=60)

vw_entire_period_table_36 = summarize_rolling_results(vw_entire_period_results_36)
vw_entire_period_table_48 = summarize_rolling_results(vw_entire_period_results_48)
vw_entire_period_table_60 = summarize_rolling_results(vw_entire_period_results_60)

In [85]:
vw_entire_period_table_36.head(10)

,parameter,mean,t_stat_NW,p_value_NW,n_obs,max_lags
0,RES,-0.038910,-2.885478,0.003908,75,3.0
1,INT,-0.035355,-2.597324,0.009395,75,3.0
2,DIV,1.361737,2.812851,0.004910,75,3.0
3,YLD,1.756160,1.924514,0.054290,75,3.0
4,TERM,-0.254773,-2.717895,0.006570,75,3.0
5,DEF,0.841626,1.412282,0.157867,75,3.0
6,Adj_R_SQUARED,0.095498,4.211384,0.000025,75,3.0
7,% RES > 0,30.666667,NaN,0.001080,75,NaN


In [86]:
vw_entire_period_table_48.head(10)

,parameter,mean,t_stat_NW,p_value_NW,n_obs,max_lags
0,RES,-0.043052,-4.814717,0.000001,63,3.0
1,INT,-0.040436,-4.351835,0.000014,63,3.0
2,DIV,1.314934,4.641497,0.000003,63,3.0
3,YLD,1.424597,1.995044,0.046038,63,3.0
4,TERM,-0.208395,-2.436448,0.014832,63,3.0
5,DEF,1.108383,2.687261,0.007204,63,3.0
6,Adj_R_SQUARED,0.099503,5.035502,0.000000,63,3.0
7,% RES > 0,12.698413,NaN,0.000000,63,NaN


In [87]:
vw_entire_period_table_60.head(10)

,parameter,mean,t_stat_NW,p_value_NW,n_obs,max_lags
0,RES,-0.044533,-11.176799,0.000000,51,3.0
1,INT,-0.042828,-7.859030,0.000000,51,3.0
2,DIV,1.090330,7.565757,0.000000,51,3.0
3,YLD,0.674897,1.565158,0.117546,51,3.0
4,TERM,-0.196933,-2.433477,0.014955,51,3.0
5,DEF,1.660220,6.759360,0.000000,51,3.0
6,Adj_R_SQUARED,0.090794,7.427045,0.000000,51,3.0
7,% RES > 0,3.921569,NaN,0.000000,51,NaN
